# Order cleaning

Cleans the raw shop export before the weekly revenue report. Run the cells top to bottom.

In [ ]:
raw_orders = [
    {"order_id": "A100", "email": "ana@example.com", "amount": "$1,300.00", "country": "us", "updated": "2024-05-03"},
    {"order_id": "A101", "email": "ben@example.com", "amount": "89.90", "country": "DE", "updated": "2024-05-02"},
    {"order_id": "A100", "email": " Ana@Example.com ", "amount": "$1,250.00", "country": "US", "updated": "2024-05-01"},
    {"order_id": "A102", "email": "dev@example.com", "amount": "", "country": "FR", "updated": "2024-05-02"},
    {"order_id": "A103", "email": "  Cara@Example.com", "amount": "45.5", "country": "gb", "updated": "2024-05-04"},
    {"order_id": "A104", "email": "eli@example.com", "amount": None, "country": "DE", "updated": "2024-05-04"},
]

## Clean

In [ ]:
from decimal import Decimal


def parse_amount(text):
    return Decimal(text.replace("$", "").replace(",", ""))


latest = {}
for row in raw_orders:
    if not row["amount"]:
        continue
    seen = latest.get(row["order_id"])
    if seen is not None and seen["updated"] >= row["updated"]:
        continue
    latest[row["order_id"]] = {
        "order_id": row["order_id"],
        "email": row["email"].strip().lower(),
        "amount": parse_amount(row["amount"]),
        "country": row["country"].upper(),
        "updated": row["updated"],
    }

clean_orders = sorted(latest.values(), key=lambda order: order["order_id"])

## Revenue by country

In [ ]:
revenue_by_country = {}
for order in clean_orders:
    country = order["country"]
    revenue_by_country[country] = revenue_by_country.get(country, Decimal("0")) + order["amount"]
revenue_by_country